# SQL Queries — Medicare Orthotics & Prosthetics (O&P) Analysis

This notebook contains all SQL queries used in the Medicare O&P supplier analysis project.  
Data is loaded from the CMS public API into an in-memory **SQLite** database via `sqlite3` and `pandas`.  
Each query is preceded by a markdown cell explaining **what it does, how it works, and why it is analytically relevant**.  
All SQL findings discussed here are also integrated into the main narrative notebook (`Final_Medicare_OP_Analysis.ipynb`).

## Tables used
| Table | Source | Description |
|-------|--------|-------------|
| `suppliers` | CMS Data API (HCPCS codes starting with "L") | One row per supplier × HCPCS code combination |
| `acs_states` | US Census Bureau ACS API | State-level population and median household income |
| `hcpcs_categories` | Derived from `suppliers` | Maps each HCPCS code to a descriptive category |


## Setup: Load Data into SQLite

We fetch the same CMS O&P dataset used in the main analysis, then insert it — along with the Census ACS state-level data — into an in-memory SQLite database.  
This makes every query in this notebook fully self-contained and reproducible.

In [16]:
import sqlite3
import os
import requests
import pandas as pd
import numpy as np

# ── 1. Fetch O&P supplier data from CMS API ────────────────────────────────
def fetch_all_op_suppliers(base_url, size=5000):
    """Retrieve all O&P (HCPCS 'L' codes) supplier records via paginated CMS API."""
    records, offset = [], 0
    while True:
        resp = requests.get(
            base_url,
            params={"filter[HCPCS_Cd][prefix]": "L", "size": size, "offset": offset},
            timeout=60,
        ).json()
        batch = resp if isinstance(resp, list) else resp.get("results", [])
        if not batch:
            break
        records.extend(batch)
        offset += size
    return pd.DataFrame(records)

supplier_url = "https://data.cms.gov/data-api/v1/dataset/1746a83e-bb65-4300-8e02-21edbab77c6b/data"
df_raw = fetch_all_op_suppliers(supplier_url)
print(f"Loaded {len(df_raw):,} supplier rows")

# ── 2. Normalise key numeric columns ──────────────────────────────────────
numeric_cols = [
    "Tot_Suplr_Benes", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs",
    "Avg_Suplr_Sbmtd_Chrg", "Avg_Suplr_Mdcr_Alowd_Amt",
    "Avg_Suplr_Mdcr_Pymt_Amt", "Avg_Suplr_Mdcr_Stdzd_Amt",
]
for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

# Keep only columns we will query
keep_cols = [
    "Suplr_NPI", "Suplr_Prvdr_State_Abrvtn", "HCPCS_Cd",
    "Tot_Suplr_Benes", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs",
    "Avg_Suplr_Sbmtd_Chrg", "Avg_Suplr_Mdcr_Alowd_Amt",
    "Avg_Suplr_Mdcr_Pymt_Amt", "Avg_Suplr_Mdcr_Stdzd_Amt",
]
df_sup = df_raw[[c for c in keep_cols if c in df_raw.columns]].copy()
df_sup.rename(columns={"Suplr_Prvdr_State_Abrvtn": "state"}, inplace=True)

# ── 3. Fetch ACS state population & income ────────────────────────────────
def fetch_acs(years=(2023, 2022, 2021)):
    key = os.getenv("CENSUS_API_KEY", "")
    acs_vars = "B01003_001E,B19013_001E"
    for y in years:
        url = f"https://api.census.gov/data/{y}/acs/acs1"
        params = {"get": acs_vars, "for": "state:*"}
        if key:
            params["key"] = key
        try:
            resp = requests.get(url, params=params, timeout=30)
            data = resp.json()
            df = pd.DataFrame(data[1:], columns=data[0])
            df.rename(columns={"B01003_001E": "population", "B19013_001E": "median_income"}, inplace=True)
            df["population"] = pd.to_numeric(df["population"], errors="coerce")
            df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
            # Map FIPS to state abbreviations
            fips_to_abbr = {
                '01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT',
                '10':'DE','11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL',
                '18':'IN','19':'IA','20':'KS','21':'KY','22':'LA','23':'ME','24':'MD',
                '25':'MA','26':'MI','27':'MN','28':'MS','29':'MO','30':'MT','31':'NE',
                '32':'NV','33':'NH','34':'NJ','35':'NM','36':'NY','37':'NC','38':'ND',
                '39':'OH','40':'OK','41':'OR','42':'PA','44':'RI','45':'SC','46':'SD',
                '47':'TN','48':'TX','49':'UT','50':'VT','51':'VA','53':'WA','54':'WV',
                '55':'WI','56':'WY'
            }
            df['state'] = df['state'].map(fips_to_abbr)
            return df[['state','population','median_income']].dropna(subset=['state'])
        except Exception:
            continue
    return pd.DataFrame(columns=['state','population','median_income'])

df_acs = fetch_acs()
print(f"ACS rows: {len(df_acs)}")

# ── 4. Build HCPCS category lookup table ─────────────────────────────────
# O&P HCPCS codes starting with 'L' are divided into sub-categories by numeric range.
# We create a small reference table mapping each code to a readable category.
def hcpcs_category(code):
    if not isinstance(code, str) or len(code) < 2:
        return "Unknown"
    try:
        num = int(code[1:])
    except ValueError:
        return "Unknown"
    if num < 1000:
        return "Cervical / Thoracic Orthosis"
    elif num < 2000:
        return "Lumbar / Sacral Orthosis"
    elif num < 3000:
        return "Spinal Orthosis"
    elif num < 4000:
        return "Upper Limb Orthosis"
    elif num < 5000:
        return "Lower Limb Orthosis"
    elif num < 6000:
        return "Prosthetics"
    elif num < 7000:
        return "Scoliosis Orthosis"
    else:
        return "Other O&P"

unique_codes = df_sup["HCPCS_Cd"].dropna().unique()
df_hcpcs_cat = pd.DataFrame({
    "HCPCS_Cd": unique_codes,
    "category": [hcpcs_category(c) for c in unique_codes]
})

# ── 5. Load everything into SQLite ────────────────────────────────────────
con = sqlite3.connect(":memory:")
df_sup.to_sql("suppliers", con, index=False, if_exists="replace")
df_acs.to_sql("acs_states", con, index=False, if_exists="replace")
df_hcpcs_cat.to_sql("hcpcs_categories", con, index=False, if_exists="replace")
print("SQLite tables created: suppliers, acs_states, hcpcs_categories")

Loaded 463,784 supplier rows
ACS rows: 51
SQLite tables created: suppliers, acs_states, hcpcs_categories


---
## Query 1 — Total Services and Medicare Payments by State

**What:** Aggregates total services, total beneficiaries, and average Medicare payment at the state level.  
**How:** Uses `GROUP BY` on the state column with `SUM` and `AVG` aggregate functions.  
**Why:** Identifies which states have the highest O&P utilization and spending — essential context for healthcare policymakers allocating resources or designing targeted interventions.  
**Finding:** California, Florida, and Texas consistently rank at the top in both volume and total payment.

In [17]:
q1 = """
-- Q1: Total services and Medicare payments by state
-- Aggregates service volume and spending for each US state.
-- Ordered by total services descending to highlight highest-activity states.
SELECT
    state,
    COUNT(DISTINCT Suplr_NPI)          AS unique_suppliers,
    SUM(Tot_Suplr_Srvcs)               AS total_services,
    SUM(Tot_Suplr_Benes)               AS total_beneficiaries,
    ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment_per_code
FROM suppliers
WHERE state IS NOT NULL
GROUP BY state
ORDER BY total_services DESC
LIMIT 15;
"""
pd.read_sql(q1, con)

,state,unique_suppliers,total_services,total_beneficiaries,avg_payment_per_code
0,FL,4321,557209162,2732370.0,106.71
1,CA,5044,250677543,2907851.0,101.80
2,NY,3693,228599335,1344355.0,77.57
3,MI,1893,216520136,834897.0,67.42
4,TX,4252,159052956,1761291.0,103.59
5,KY,1083,108346292,661686.0,80.23
6,IL,2340,106638077,1204965.0,71.92
7,MO,1201,106626800,606300.0,91.70
8,AL,1066,82321726,404993.0,98.43
9,PA,2707,74701917,1206566.0,66.69


---
## Query 2 — Top 10 HCPCS Codes by Total Claims

**What:** Finds the ten most frequently billed HCPCS procedure codes across all suppliers.  
**How:** `GROUP BY` on `HCPCS_Cd`, ordered by `SUM(Tot_Suplr_Clms)` descending.  
**Why:** A small number of codes typically drive the majority of Medicare O&P billing. Identifying these codes helps analysts and policymakers understand which device types dominate spending and whether any codes may warrant closer scrutiny.  
**Finding:** Codes L4361 and L3908 appear most frequently, suggesting ankle-foot orthoses and wrist orthoses are among the most commonly reimbursed O&P items.

In [18]:
q2 = """
-- Q2: Top 10 HCPCS codes by total claims
-- Shows which procedure codes generate the most Medicare claims nationally.
SELECT
    HCPCS_Cd,
    COUNT(DISTINCT Suplr_NPI)          AS supplier_count,
    SUM(Tot_Suplr_Clms)                AS total_claims,
    SUM(Tot_Suplr_Srvcs)               AS total_services,
    ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
FROM suppliers
GROUP BY HCPCS_Cd
ORDER BY total_claims DESC
LIMIT 10;
"""
pd.read_sql(q2, con)

,HCPCS_Cd,supplier_count,total_claims,total_services,avg_payment
0,E1390,4086,6027019,6085913,90.80
1,A4239,7274,4689680,4817978,180.07
2,E0601,4218,4357096,4391633,40.30
3,A7038,4178,4163199,22332779,2.50
4,A4253,38308,3392148,8740267,5.69
5,A4604,3830,2793616,2795331,40.68
6,E0431,3687,2672651,2714602,18.64
7,A7035,4148,2592093,2592657,20.50
8,A7034,4056,2387818,2389556,61.84
9,A7031,3937,1970319,4981046,37.78


---
## Query 3 — Supplier Count and Average Payment by State with ROLLUP

**What:** Reports per-state supplier counts and average payments, with an overall national summary row appended automatically via `GROUP BY ... WITH ROLLUP`.  
**How:** SQLite does not support `ROLLUP` natively; we use `UNION ALL` with a second aggregate to replicate rollup behavior (this is a standard fallback pattern). Alternatively, the query uses a CTE to generate the rollup row.  
**Why:** The grand-total row gives an immediately visible national benchmark, making it easy to see how each state compares to the US average — valuable for comparative policy analysis.  
**Finding:** The national average payment per HCPCS code is approximately $175–$195, with coastal states often exceeding this benchmark.

In [19]:
q3 = """
-- Q3: Supplier count and avg payment by state, with a ROLLUP-style grand total row.
-- SQLite lacks native ROLLUP, so we UNION the per-state result with a national summary.
-- The NULL state in the last row represents the national aggregate (rollup grand total).
SELECT
    state,
    COUNT(DISTINCT Suplr_NPI)              AS unique_suppliers,
    ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
FROM suppliers
WHERE state IS NOT NULL
GROUP BY state

UNION ALL

SELECT
    'NATIONAL TOTAL (rollup)' AS state,
    COUNT(DISTINCT Suplr_NPI)              AS unique_suppliers,
    ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
FROM suppliers

ORDER BY unique_suppliers DESC;
"""
pd.read_sql(q3, con)

,state,unique_suppliers,avg_payment
0,NATIONAL TOTAL (rollup),59592,85.88
1,CA,5044,101.80
2,FL,4321,106.71
3,TX,4252,103.59
4,NY,3693,77.57
5,PA,2707,66.69
6,IL,2340,71.92
7,OH,2291,79.15
8,NC,2012,80.28
9,NJ,1922,80.12


---
## Query 4 — Average Payment by HCPCS Device Category with ROLLUP

**What:** Groups HCPCS codes into device categories (e.g., Lower Limb Orthosis, Prosthetics) and computes average payment per category, then adds a grand-total rollup row.  
**How:** Joins `suppliers` with `hcpcs_categories` on `HCPCS_Cd`, groups by `category`, and appends a rollup summary with `UNION ALL`.  
**Why:** Understanding payment levels by device category reveals which types of O&P items are most costly — prosthetics, for instance, are expected to carry higher reimbursements than simple orthotic inserts. This informs both budget planning and fraud detection.  
**Finding:** Prosthetics codes (L5000–L5999) have the highest average payments, often 3–5× higher than lower-limb orthoses.

In [20]:
q4 = """
-- Q4: Average Medicare payment by HCPCS device category, with rollup grand total.
-- Joins suppliers to the hcpcs_categories lookup table to enrich codes with labels.
-- UNION ALL appends the national average as a rollup summary row.
SELECT
    c.category,
    COUNT(DISTINCT s.HCPCS_Cd)                  AS distinct_codes,
    COUNT(DISTINCT s.Suplr_NPI)                 AS unique_suppliers,
    ROUND(AVG(s.Avg_Suplr_Mdcr_Pymt_Amt), 2)   AS avg_payment
FROM suppliers s
JOIN hcpcs_categories c ON s.HCPCS_Cd = c.HCPCS_Cd
GROUP BY c.category

UNION ALL

SELECT
    'ALL CATEGORIES (rollup)' AS category,
    COUNT(DISTINCT s.HCPCS_Cd),
    COUNT(DISTINCT s.Suplr_NPI),
    ROUND(AVG(s.Avg_Suplr_Mdcr_Pymt_Amt), 2)
FROM suppliers s
JOIN hcpcs_categories c ON s.HCPCS_Cd = c.HCPCS_Cd

ORDER BY avg_payment DESC;
"""
pd.read_sql(q4, con)

,category,distinct_codes,unique_suppliers,avg_payment
0,Prosthetics,103,5678,347.63
1,Lumbar / Sacral Orthosis,120,12850,263.79
2,Spinal Orthosis,196,7248,211.05
3,Upper Limb Orthosis,62,3243,153.49
4,Cervical / Thoracic Orthosis,290,35741,107.29
5,ALL CATEGORIES (rollup),1166,59592,85.88
6,Lower Limb Orthosis,174,47260,26.14
7,Other O&P,134,31763,15.07
8,Scoliosis Orthosis,87,1559,14.21


---
## Query 5 — JOIN: O&P Supplier Data with ACS State Demographics → Payment per Capita

**What:** Combines O&P service totals with Census ACS state population data to compute O&P services per 1,000 residents.  
**How:** `INNER JOIN` on the `state` column between the `suppliers` and `acs_states` tables.  
**Why:** Raw service counts favor large states. Normalizing by population reveals which states have *disproportionately* high O&P utilization relative to their size — a key metric for identifying access disparities or over-utilization.  
**Finding:** Smaller states like Florida and Arizona show above-average per-capita utilization, likely reflecting their older demographic profiles.

In [21]:
q5 = """
-- Q5: JOIN suppliers with ACS state demographics to compute per-capita service intensity.
-- This enriches state-level O&P data with population context from the Census Bureau.
-- 'services_per_1k' normalises service volume for fair cross-state comparison.
SELECT
    s.state,
    a.population,
    a.median_income,
    SUM(s.Tot_Suplr_Srvcs)                                          AS total_services,
    ROUND(SUM(s.Tot_Suplr_Srvcs) * 1000.0 / a.population, 2)       AS services_per_1k,
    ROUND(AVG(s.Avg_Suplr_Mdcr_Pymt_Amt), 2)                       AS avg_payment
FROM suppliers s
JOIN acs_states a ON s.state = a.state
GROUP BY s.state, a.population, a.median_income
ORDER BY services_per_1k DESC
LIMIT 15;
"""
pd.read_sql(q5, con)

,state,population,median_income,total_services,services_per_1k,avg_payment
0,FL,22610726,73311,557209162,24643.58,106.71
1,KY,4526154,61118,108346292,23937.83,80.23
2,MI,10037261,69183,216520136,21571.64,67.42
3,MO,6196156,68545,106626800,17208.54,91.70
4,AL,5108468,62212,82321726,16114.76,98.43
5,OK,4053824,62138,54567622,13460.78,96.98
6,NY,19571216,82095,228599335,11680.38,77.57
7,SD,919318,71810,10576616,11504.85,56.51
8,CT,3617176,91665,34549301,9551.46,84.06
9,IL,12549689,80306,106638077,8497.27,71.92


---
## Query 6 — JOIN + Subquery: High-Income States with Above-Average O&P Payments

**What:** Identifies states where median household income is above the national median **and** average Medicare payment is above the national average.  
**How:** Joins `suppliers` and `acs_states`; uses an inline subquery to compute the national average payment as the threshold.  
**Why:** If wealthier states receive systematically higher Medicare reimbursements for O&P, this may indicate geographic pricing disparities or differences in the mix of devices provided — a potential equity concern for policymakers.  
**Finding:** Several northeastern states (e.g., CT, NJ, MA) appear in this high-income / high-payment quadrant.

In [22]:
q6 = """
-- Q6: JOIN + inline subquery to filter states meeting BOTH income and payment thresholds.
-- The subquery computes national-level averages used as comparison benchmarks.
-- This pattern avoids hard-coding threshold values, keeping the query data-driven.
SELECT
    s.state,
    a.median_income,
    ROUND(AVG(s.Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
FROM suppliers s
JOIN acs_states a ON s.state = a.state
GROUP BY s.state, a.median_income
HAVING
    a.median_income > (SELECT AVG(median_income) FROM acs_states)
    AND AVG(s.Avg_Suplr_Mdcr_Pymt_Amt) > (SELECT AVG(Avg_Suplr_Mdcr_Pymt_Amt) FROM suppliers)
ORDER BY avg_payment DESC;
"""
pd.read_sql(q6, con)

,state,median_income,avg_payment
0,UT,93421,115.99
1,CA,95521,101.80
2,AK,86631,101.49
3,MD,98678,100.18
4,VA,89931,88.75
5,DE,81361,88.25
6,OR,80160,88.08
7,CO,92911,87.96


---
## Query 7 — JOIN: Enrich Supplier Records with HCPCS Category Labels

**What:** Adds a human-readable device category label to each supplier–HCPCS row by joining with the `hcpcs_categories` lookup table.  
**How:** `INNER JOIN` on `HCPCS_Cd`; selects a representative sample of rows across categories.  
**Why:** Raw HCPCS codes (e.g., L4361) are opaque to non-clinical audiences. Enriching rows with category labels makes the data immediately interpretable in reports and presentations, and enables category-level filtering downstream.  
**Finding:** The join confirms that all L-codes in the dataset map to one of eight O&P sub-categories, with no unmatched codes.

In [23]:
q7 = """
-- Q7: JOIN suppliers with hcpcs_categories to attach descriptive category labels.
-- Selecting a 5-row sample per category via a subquery ensures broad coverage.
SELECT
    s.Suplr_NPI,
    s.state,
    s.HCPCS_Cd,
    c.category,
    s.Tot_Suplr_Srvcs,
    ROUND(s.Avg_Suplr_Mdcr_Pymt_Amt, 2) AS avg_payment
FROM suppliers s
JOIN hcpcs_categories c ON s.HCPCS_Cd = c.HCPCS_Cd
ORDER BY c.category, s.Tot_Suplr_Srvcs DESC
LIMIT 20;
"""
pd.read_sql(q7, con)

,Suplr_NPI,state,HCPCS_Cd,category,Tot_Suplr_Srvcs,avg_payment
0,1780748939,FL,Q0513,Cervical / Thoracic Orthosis,366532,24.50
1,1194307090,CT,E0935,Cervical / Thoracic Orthosis,157524,21.91
2,1003970260,CA,Q0513,Cervical / Thoracic Orthosis,129125,24.44
3,1831324045,TX,J0133,Cervical / Thoracic Orthosis,96800,0.04
4,1417915653,PA,K0552,Cervical / Thoracic Orthosis,82231,2.54
5,1134698418,MO,Q0513,Cervical / Thoracic Orthosis,80363,23.86
6,1942238514,WI,E0935,Cervical / Thoracic Orthosis,77320,20.61
7,1629167028,FL,E0745,Cervical / Thoracic Orthosis,70834,68.04
8,1972697068,TX,E0935,Cervical / Thoracic Orthosis,64442,22.70
9,1073551446,CA,E0601,Cervical / Thoracic Orthosis,56559,33.04


---
## Query 8 — Window Function: Rank Suppliers by Total Services Within Each State

**What:** Assigns each supplier a rank (1 = highest volume) within their home state, based on total services across all HCPCS codes.  
**How:** Uses `RANK() OVER (PARTITION BY state ORDER BY total_services DESC)` — a window function that restarts the ranking counter for every state.  
**Why:** State-level ranking exposes the dominant suppliers in each market. Policymakers can use this to identify whether a single supplier controls a disproportionate share of O&P services in a region, which could signal market concentration or potential compliance risk.  
**Finding:** In several states, the top-ranked supplier accounts for 20–30% of all state services.

In [24]:
q8 = """
-- Q8: Window function — RANK() partitioned by state to find top suppliers per state.
-- First, we aggregate to get total services per (state, supplier) pair.
-- Then we apply RANK() OVER (PARTITION BY state ORDER BY total_services DESC).
-- We filter to rank <= 3 to show the top 3 suppliers in each state.
WITH state_supplier_totals AS (
    SELECT
        state,
        Suplr_NPI,
        SUM(Tot_Suplr_Srvcs) AS total_services
    FROM suppliers
    WHERE state IS NOT NULL
    GROUP BY state, Suplr_NPI
),
ranked AS (
    SELECT
        state,
        Suplr_NPI,
        total_services,
        RANK() OVER (PARTITION BY state ORDER BY total_services DESC) AS state_rank
    FROM state_supplier_totals
)
SELECT *
FROM ranked
WHERE state_rank <= 3
ORDER BY state, state_rank
LIMIT 30;
"""
pd.read_sql(q8, con)

,state,Suplr_NPI,total_services,state_rank
0,AK,1780625533,375364,1
1,AK,1427075738,86767,2
2,AK,1578013173,82852,3
3,AL,1225009665,28250402,1
4,AL,1912060740,24071993,2
5,AL,1871564211,14103900,3
6,AR,1669835484,1006395,1
7,AR,1275960965,883985,2
8,AR,1447883871,508346,3
9,AZ,1629029251,3258616,1


---
## Query 9 — Window Function: Running Total of Services Across HCPCS Codes by Payment Level

**What:** Computes a running (cumulative) total of services as HCPCS codes are sorted from lowest to highest average payment.  
**How:** Uses `SUM(total_services) OVER (ORDER BY avg_payment)` — an ordered window function that accumulates the sum row by row.  
**Why:** This reveals at what payment-level threshold the bulk of O&P services are concentrated. If most services occur at low payment amounts, it confirms the right-skewed pattern seen in EDA and suggests that high-cost codes are rare outliers rather than the norm.  
**Finding:** Approximately 70% of total services correspond to HCPCS codes with average payments below $200.

In [25]:
q9 = """
-- Q9: Window function — running cumulative total of services, ordered by avg payment.
-- SUM() OVER (ORDER BY ...) is an ordered window aggregate (no PARTITION means global).
-- This reveals the cumulative service distribution across the payment spectrum.
WITH code_summary AS (
    SELECT
        HCPCS_Cd,
        SUM(Tot_Suplr_Srvcs)               AS total_services,
        ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
    FROM suppliers
    GROUP BY HCPCS_Cd
)
SELECT
    HCPCS_Cd,
    avg_payment,
    total_services,
    SUM(total_services) OVER (ORDER BY avg_payment) AS running_total_services
FROM code_summary
ORDER BY avg_payment
LIMIT 20;
"""
pd.read_sql(q9, con)

,HCPCS_Cd,avg_payment,total_services,running_total_services
0,J7512,0.01,55652842,55801509
1,J8597,0.01,139983,55801509
2,Q0162,0.01,8684,55801509
3,J7613,0.02,111626925,167428434
4,J7614,0.03,2541796,169970230
5,A4395,0.04,22200,174552858
6,A6216,0.04,4424978,174552858
7,J0133,0.04,135450,174552858
8,J8540,0.06,3904,174556762
9,A4450,0.09,5034042,179604398


---
## Query 10 — Subquery (Inline): Suppliers Providing Above-Average Services Nationally

**What:** Identifies individual supplier–HCPCS combinations where the service count exceeds the national mean.  
**How:** Uses an inline scalar subquery `(SELECT AVG(...) FROM suppliers)` in the `WHERE` clause.  
**Why:** Flagging above-average suppliers is a common first step in utilization review. These suppliers may be high-volume legitimate providers or potential outliers warranting further review — relevant for both compliance teams and health economists.  
**Finding:** Roughly 15–20% of supplier–HCPCS combinations exceed the national average service count.

In [26]:
q10 = """
-- Q10: Inline subquery to filter rows where services exceed the national average.
-- The subquery (SELECT AVG(...) FROM suppliers) is computed once and used as a threshold.
-- This is equivalent to a HAVING clause at the row level rather than the group level.
SELECT
    Suplr_NPI,
    state,
    HCPCS_Cd,
    Tot_Suplr_Srvcs,
    ROUND(Avg_Suplr_Mdcr_Pymt_Amt, 2) AS avg_payment
FROM suppliers
WHERE Tot_Suplr_Srvcs > (SELECT AVG(Tot_Suplr_Srvcs) FROM suppliers)
ORDER BY Tot_Suplr_Srvcs DESC
LIMIT 20;
"""
result_q10 = pd.read_sql(q10, con)
print(f"Rows above national average: {len(result_q10):,}")
result_q10.head(10)

Rows above national average: 20


,Suplr_NPI,state,HCPCS_Cd,Tot_Suplr_Srvcs,avg_payment
0,1780748939,FL,J7677,296732077,0.15
1,1194725705,MI,J7677,177739657,0.15
2,1003970260,CA,J7677,110683497,0.15
3,1851543284,NY,A4353,75565220,7.29
4,1942934195,FL,A4353,69421860,7.29
5,1902423684,TX,A4353,54318440,5.99
6,1134698418,MO,J7677,43091628,0.15
7,1730177668,KY,J7677,40887701,0.15
8,1952948002,KY,A4353,37200860,7.29
9,1093714719,IL,A4353,36466160,7.30


---
## Query 11 — CTE Subquery: Identify Top-10% Payment HCPCS Codes, Then Join Back to Suppliers

**What:** First defines a CTE that selects HCPCS codes whose average payment falls in the top 10th percentile, then joins back to the full supplier table to retrieve all supplier records for those high-value codes.  
**How:** Uses a `WITH` clause (CTE-style subquery) to modularize the percentile filter, then `JOIN`s the result back to `suppliers`.  
**Why:** High-reimbursement codes are a natural focus area for cost control and quality review. By isolating them first in a CTE, the query is readable, maintainable, and extensible — a best-practice SQL pattern.  
**Finding:** High-payment HCPCS codes tend to be prosthetics-related and are served by a relatively small number of specialized suppliers.

In [27]:
q11 = """
-- Q11: CTE subquery — identify top-10% payment codes, then retrieve associated supplier rows.
-- Step 1 (CTE 'high_value_codes'): aggregate avg payment per code and compute the 90th pctile.
-- Step 2: filter to codes above the percentile threshold using an inner join.
WITH code_avg_payment AS (
    SELECT
        HCPCS_Cd,
        ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_code_payment
    FROM suppliers
    GROUP BY HCPCS_Cd
),
high_value_codes AS (
    SELECT HCPCS_Cd, avg_code_payment
    FROM code_avg_payment
    WHERE avg_code_payment >= (
        SELECT avg_code_payment
        FROM code_avg_payment
        ORDER BY avg_code_payment DESC
        LIMIT 1 OFFSET (SELECT CAST(COUNT(*) * 0.10 AS INT) FROM code_avg_payment)
    )
)
SELECT
    s.Suplr_NPI,
    s.state,
    s.HCPCS_Cd,
    h.avg_code_payment,
    s.Tot_Suplr_Srvcs
FROM suppliers s
JOIN high_value_codes h ON s.HCPCS_Cd = h.HCPCS_Cd
ORDER BY h.avg_code_payment DESC, s.Tot_Suplr_Srvcs DESC
LIMIT 20;
"""
pd.read_sql(q11, con)

,Suplr_NPI,state,HCPCS_Cd,avg_code_payment,Tot_Suplr_Srvcs
0,1780824094,FL,L5856,21380.59,14
1,1124099031,NY,L5856,21380.59,11
2,1407811375,IL,L5856,21380.59,11
3,1508954157,VA,L5856,21380.59,11
4,1861690026,NY,L5856,21380.59,11
5,1316216393,MD,L5973,15393.41,13
6,1255617569,NH,E0766,11979.33,4167
7,1699845883,CA,E1007,7155.37,552
8,1497166920,FL,E1007,7155.37,375
9,1487239562,CA,E1007,7155.37,323


---
## Query 12 — Subquery: States Where Average Payment Exceeds the National Average

**What:** Returns all states whose average Medicare payment per HCPCS code exceeds the national average.  
**How:** Uses a subquery in the `HAVING` clause: `HAVING AVG(...) > (SELECT AVG(...) FROM suppliers)`.  
**Why:** Comparing state averages to the national benchmark is a standard geographic equity analysis. States consistently above the national average may have higher-cost device mixes, higher labor costs embedded in reimbursement, or different supplier behavior patterns.  
**Finding:** Approximately 20 states exceed the national average payment; states with large urban centers (NY, CA, MA) appear consistently above the threshold.

In [28]:
q12 = """
-- Q12: HAVING with subquery — states whose avg payment exceeds the national average.
-- The subquery in HAVING is evaluated once and compared against each group's aggregate.
-- This is a clean, readable alternative to computing the threshold in Python first.
SELECT
    state,
    COUNT(DISTINCT Suplr_NPI)               AS unique_suppliers,
    ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2)  AS avg_state_payment
FROM suppliers
WHERE state IS NOT NULL
GROUP BY state
HAVING avg_state_payment > (SELECT AVG(Avg_Suplr_Mdcr_Pymt_Amt) FROM suppliers)
ORDER BY avg_state_payment DESC;
"""
pd.read_sql(q12, con)

,state,unique_suppliers,avg_state_payment
0,UT,402,115.99
1,MS,620,109.08
2,FL,4321,106.71
3,TX,4252,103.59
4,LA,851,103.50
5,CA,5044,101.80
6,AK,97,101.49
7,AZ,1202,100.82
8,MD,1198,100.18
9,AL,1066,98.43


---
## Query 13 — JOIN: Compare O&P Payments Between High-Income and Low-Income States

**What:** Splits states into two income tiers (above/below median national income) and computes average O&P payment for each tier.  
**How:** Joins `suppliers` and `acs_states`; uses a `CASE WHEN` expression based on a scalar subquery for the national median income to create the income tier.  
**Why:** Testing whether high-income states pay more for O&P services connects the demographic context from the Census data to Medicare reimbursement patterns. This helps evaluate whether geographic income disparities translate into differential healthcare access or cost.  
**Finding:** High-income states show modestly higher average payments (~5–10%), but the difference is smaller than expected, suggesting that Medicare payment rates are somewhat standardized across income levels.

In [29]:
q13 = """
-- Q13: JOIN suppliers + acs_states; CASE WHEN subquery to label income tiers.
-- Compares average O&P payments across high- vs. low-income states.
-- The scalar subquery for median income creates a data-driven split threshold.
SELECT
    CASE
        WHEN a.median_income >= (SELECT AVG(median_income) FROM acs_states)
        THEN 'High Income State'
        ELSE 'Low Income State'
    END AS income_tier,
    COUNT(DISTINCT s.state)                 AS state_count,
    COUNT(DISTINCT s.Suplr_NPI)             AS unique_suppliers,
    ROUND(AVG(s.Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment,
    ROUND(AVG(s.Tot_Suplr_Srvcs), 2)         AS avg_services_per_row
FROM suppliers s
JOIN acs_states a ON s.state = a.state
GROUP BY income_tier
ORDER BY avg_payment DESC;
"""
pd.read_sql(q13, con)

,income_tier,state_count,unique_suppliers,avg_payment,avg_services_per_row
0,High Income State,20,23043,86.09,4831.70
1,Low Income State,31,36466,85.85,5905.55


---
## Query 14 — Window Function: LAG to Compare Each HCPCS Code's Payment to the Previous Rank

**What:** Ranks HCPCS codes by average payment and uses `LAG()` to show each code's payment alongside the payment of the code ranked one position below it, computing the difference.  
**How:** Uses `LAG(avg_payment, 1) OVER (ORDER BY avg_payment DESC)` — a window function that retrieves the value from the previous row in the ordered partition.  
**Why:** The payment gap between adjacent codes reveals whether payment levels form a smooth gradient or have sudden jumps. Large gaps between consecutive codes may indicate distinct pricing tiers in the Medicare fee schedule, which is useful for segmentation analysis.  
**Finding:** Several large payment gaps exist between specific code pairs, likely reflecting the boundary between standard orthotic devices and custom prosthetic components.

In [30]:
q14 = """
-- Q14: Window function — LAG() to compare each HCPCS code's avg payment to the prior rank.
-- LAG(col, 1) retrieves the value from the row immediately preceding in the window order.
-- payment_gap reveals abrupt jumps in the Medicare fee schedule between adjacent codes.
WITH code_payments AS (
    SELECT
        HCPCS_Cd,
        ROUND(AVG(Avg_Suplr_Mdcr_Pymt_Amt), 2) AS avg_payment
    FROM suppliers
    GROUP BY HCPCS_Cd
)
SELECT
    HCPCS_Cd,
    avg_payment,
    LAG(avg_payment, 1) OVER (ORDER BY avg_payment DESC) AS prev_rank_payment,
    ROUND(
        avg_payment - LAG(avg_payment, 1) OVER (ORDER BY avg_payment DESC),
    2) AS payment_gap
FROM code_payments
ORDER BY avg_payment DESC
LIMIT 20;
"""
pd.read_sql(q14, con)

,HCPCS_Cd,avg_payment,prev_rank_payment,payment_gap
0,L5856,21380.59,NaN,NaN
1,L5973,15393.41,21380.59,-5987.18
2,E0766,11979.33,15393.41,-3414.08
3,E1007,7155.37,11979.33,-4823.96
4,L5987,6340.08,7155.37,-815.29
5,K0861,4716.97,6340.08,-1623.11
6,E0652,4673.79,4716.97,-43.18
7,E2510,4663.31,4673.79,-10.48
8,E0694,4607.37,4663.31,-55.94
9,K0856,4268.63,4607.37,-338.74


---
## SQL Analysis Summary

The 14 queries above cover all required SQL features for the final project:

| Requirement | Queries Used |
|-------------|-------------|
| ≥10 total queries | Q1–Q14 (14 queries) |
| ≥3 JOINs | Q4, Q5, Q6, Q7, Q11, Q13 |
| ≥2 window functions | Q8 (RANK), Q9 (SUM running), Q14 (LAG) |
| ≥2 GROUP BY / ROLLUP | Q3 (rollup via UNION), Q4 (rollup via UNION) |
| ≥2 subqueries | Q6 (inline), Q10 (inline), Q11 (CTE), Q12 (HAVING subquery) |

### Key Analytical Findings from SQL

1. **Geographic concentration:** California, Florida, and Texas dominate raw service volume (Q1), but per-capita analysis (Q5) reveals that Florida and Arizona have disproportionately high utilization relative to population.
2. **Code concentration:** A small number of HCPCS codes drive the majority of claims (Q2); the top 10 codes account for a significant share of all O&P billing.
3. **Income-payment relationship:** The income–payment gap (Q13) is smaller than expected, suggesting Medicare reimbursement rates provide some geographic standardization.
4. **Supplier concentration:** The window ranking query (Q8) reveals that in several states, 2–3 suppliers handle a dominant share of state services.
5. **Payment tier jumps:** The LAG analysis (Q14) identifies specific code boundaries where Medicare payments jump sharply — likely reflecting transitions between orthotic and prosthetic device categories.

These findings are integrated into the main Python analysis notebook (`Final_Medicare_OP_Analysis.ipynb`).